# A tour of `stochastic_lob`

Nine short examples, one per question a liquidity provider asks. Each cell runs in seconds. Units: time in seconds, prices in ticks, order counts in orders, unless a cell says otherwise.

In [1]:
import sys, numpy as np
sys.path.insert(0, '../src')
from stochastic_lob import best_queues, queue_position, market_making as mm, amm, hawkes

## 1. Where will the price move next?

Cont and de Larrard: with 2 orders at the best bid and 5 at the best ask, and equal arrival and depletion rates, the probability that the next mid-price move is up, exactly and by the arctangent law.

In [2]:
for qb, qa in [(2, 5), (5, 2), (10, 10), (40, 20)]:
    print(qb, qa, round(best_queues.p_up_balanced(qb, qa), 4), round(best_queues.p_up_arctan(qb, qa), 4))

2 5 0.244 0.2422
5 2 0.756 0.7578
10 10 0.5 0.5
40 20 0.7048 0.7048


## 2. What is my place in the queue worth?

One sell order at the best ask, a queue of 8 orders, informed sweeps at rate 0.03 per second that move fair value by 2 ticks. Fill probability and markout per fill (ticks) by number of orders ahead.

In [3]:
r = queue_position.QueueRates(lam=1.0, mu=0.9, theta=0.01, nu=0.03, jump=2.0)
V = queue_position.solve_queue_value(r, a_max=7, b_max=30, c_max=30)
c = 8
for a in range(8):
    b = 7 - a
    p = V.p_fill[a, b, c]
    print(f'ahead {a}: P(fill) {p:.3f}, markout per fill {V.value[a, b, c] / p:+.3f}')

ahead 0: P(fill) 0.968, markout per fill +0.432
ahead 1: P(fill) 0.938, markout per fill +0.364
ahead 2: P(fill) 0.909, markout per fill +0.296
ahead 3: P(fill) 0.881, markout per fill +0.227
ahead 4: P(fill) 0.853, markout per fill +0.158
ahead 5: P(fill) 0.827, markout per fill +0.089
ahead 6: P(fill) 0.802, markout per fill +0.020
ahead 7: P(fill) 0.778, markout per fill -0.049


## 3. How should I quote with an inventory limit?

Exact GLFT quotes (ticks from the mid) for inventory from -3 to 3, long horizon, with and without a positive drift.

In [4]:
p = mm.MMParams(S0=0.0, T=600.0, sigma=0.3, dt=1.0, gamma=0.01, k=0.3, A=0.9)
for drift in (0.0, 0.005):
    db, da = mm.glft_exact_quotes(30, p, [600.0], drift)
    print('drift', drift)
    for y in range(-3, 4):
        print(f'  y={y:+d}: bid {db[0, y + 30]:.3f}  ask {da[0, y + 30]:.3f}')

drift 0.0
  y=-3: bid 3.109  ask 3.517
  y=-2: bid 3.177  ask 3.449
  y=-1: bid 3.245  ask 3.381
  y=+0: bid 3.313  ask 3.313
  y=+1: bid 3.381  ask 3.245
  y=+2: bid 3.449  ask 3.177
  y=+3: bid 3.517  ask 3.109
drift 0.005
  y=-3: bid 2.731  ask 3.895
  y=-2: bid 2.799  ask 3.827
  y=-1: bid 2.867  ask 3.759
  y=+0: bid 2.935  ask 3.691
  y=+1: bid 3.003  ask 3.623
  y=+2: bid 3.071  ask 3.555
  y=+3: bid 3.139  ask 3.487


## 4. Is my order flow self-exciting?

Simulate a univariate Hawkes process, fit it by maximum likelihood, and read the branching ratio.

In [5]:
true = hawkes.ExpHawkes([0.5], [[1.0]], [[2.5]])
t, ty = true.simulate(4000, rng=7)
fit, se, _ = hawkes.fit_exp_hawkes(t, ty, 4000, 1)
print('events', len(t))
print('branching ratio', fit.alpha[0, 0] / fit.beta[0, 0], 'true', 0.4)

events 3350
branching ratio 0.40647212280936024 true 0.4


## 5. What does an AMM position cost?

Loss-versus-rebalancing per unit of capital for a constant-product pool and for concentrated ranges, with daily volatility 4% (time unit: one day).

In [6]:
sigma = 0.04
print('full range, % per year:', 100 * 365 * sigma**2 / 8)
for w in (2.0, 1.2, 1.05):
    print(f'range [P/{w}, P*{w}], % per year:', round(100 * 365 * amm.lvr_per_capital_concentrated(sigma, w), 1))

full range, % per year: 7.300000000000001
range [P/2.0, P*2.0], % per year: 24.9
range [P/1.2, P*1.2], % per year: 83.8
range [P/1.05, P*1.05], % per year: 302.9


## 6. How big is the impact of a metaorder?

Latent order book with slope $L = 1$ share/tick$^2$ and diffusion $D = 1$ tick$^2$/s. Peak impact (ticks) of $Q$ shares executed over $T = 1$ s, from the closed form, against the linear limit $Q/(L\sqrt{\pi D T})$ and the square-root limit $\sqrt{2Q/L}$ (deep dive 5).

In [7]:
from stochastic_lob import latent_book
for Q in (0.01, 1.0, 100.0, 1e4):
    I = latent_book.impact_peak(Q, 1.0, 1.0, 1.0)
    print(f'Q={Q:>8g}  I={I:9.4f}  linear={Q/np.sqrt(np.pi):9.4f}  sqrt={np.sqrt(2*Q):8.4f}')

Q=    0.01  I=   0.0056  linear=   0.0056  sqrt=  0.1414
Q=       1  I=   0.5581  linear=   0.5642  sqrt=  1.4142
Q=     100  I=  13.9325  linear=  56.4190  sqrt= 14.1421
Q=   10000  I= 141.4001  linear=5641.8958  sqrt=141.4214


## 7. Is that branching ratio real?

A pure Poisson process (no excitation) with a U-shaped intraday rate, fitted with an exponential Hawkes model, before and after a time change by the known profile (deep dive 4).

In [8]:
DAY = 23400.0
rate = lambda s: 0.25 * (1 + 3 * (2 * (s % DAY) / DAY - 1) ** 2)
t = hawkes.simulate_inhomogeneous_poisson(rate, DAY, 1.0, rng=3)
print('raw n_hat:', round(hawkes.fit_exp_hawkes_1d(t, DAY)[1], 3))
tt, Tt = hawkes.time_change(t, rate, np.linspace(0, DAY, 20001))
print('time-changed n_hat:', round(hawkes.fit_exp_hawkes_1d(tt, Tt)[1], 3))

raw n_hat: 0.916


time-changed n_hat: 0.008


## 8. How fast do my quotes fill?

Quote spells at distances 0 to 15 ticks, kept for at most 30 s. The exposure likelihood handles censored spells; the naive fill-ratio fit does not (deep dive 7). True $A = 0.9$/s, $k = 0.3$/tick.

In [9]:
d = np.repeat(np.arange(0.0, 16.0), 400)
tau, filled = mm.simulate_quote_spells(0.9, 0.3, d, 30.0, rng=5)
A, k, cov = mm.fit_fill_intensity(d, tau, filled)
print(f'MLE:   A={A:.3f} +- {A*np.sqrt(cov[0,0]):.3f}, k={k:.3f} +- {np.sqrt(cov[1,1]):.3f}')
print('naive: A=%.3f, k=%.3f' % mm.fit_fill_ratio_naive(d, filled, 30.0))

MLE:   A=0.934 +- 0.024, k=0.302 +- 0.003
naive: A=0.085, k=0.139


## 9. Can I run this on real data?

Write a simulated day in the LOBSTER format, read it back as real data would arrive, and estimate the book's rates with standard errors (deep dive 1 and `docs/data.md`).

In [10]:
import tempfile
from stochastic_lob import lobster, markov_lob as ml
p = ml.power_law_params(L=30, lam1=1.9, a=0.5, theta0=0.5, mu=1.0)
rec = ml.simulate_markov_lob(p, 40_000, rng=1, burn_in=5000, record_levels=40)
with tempfile.TemporaryDirectory() as tmp:
    msg, book = lobster.write(rec, tmp, levels=10)
    rec2 = lobster.to_record(lobster.read(msg, book), K=40)
est = ml.estimate_rates(rec2, L=5)
print('lam(i):', np.round(est['lam'], 3), ' true:', np.round(p.lam[:5], 3))
print('mu: %.3f +- %.3f (true 1.0)' % (est['mu'], est['se_mu']))

lam(i): [1.826 1.334 1.122 0.993 0.878]  true: [1.9   1.344 1.097 0.95  0.85 ]
mu: 0.985 +- 0.030 (true 1.0)


The slides in `slides/` put each of these in context, `docs/topics/` explains them, and `experiments/` reproduces every figure.